# 04.1 - Supervised Learning: Ekstraksi Aturan Bisnis dengan Decision Tree

**Tujuan Analisis:**
Algoritma *Decision Tree* (Pohon Keputusan) dipilih bukan hanya untuk mencari tingkat akurasi tertinggi, melainkan karena sifatnya yang transparan (mirip *flowchart* atau *White-Box*). 

Pada tahap ini, kita menggunakan **Standard K-Means** (sebagai metode *clustering* terbaik dari pengujian unsupervised) sebagai target klasifikasi. Tujuan utama dari *notebook* ini adalah untuk:
1. Melatih model agar mengenali pola dari setiap kelompok pelanggan.
2. **Mengekstrak Aturan Bisnis (Business Rules)** dalam bentuk logika *If-Else*. Aturan ini sangat vital untuk diserahkan kepada tim *Marketing* agar mereka tahu persis batasan nilai (seperti jumlah hari belanja atau total pengeluaran) dari masing-masing tipe pelanggan.

**Alur Kerja (Input & Output):**
* **Input:** Dataset `hasildata_kmeans-standard.csv` (11 Fitur Numerik dan 1 Target Cluster).
* **Output:** *Classification Report* dan Teks Aturan Bisnis.

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: Standard K-Means)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-standard.csv')

# Pisahkan Fitur dan Target
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TRAINING MODEL DECISION TREE
# Batasi kedalaman maksimal agar aturan bisnis tidak terlalu rumit dibaca
model_dt = DecisionTreeClassifier(random_state=42, max_depth=4)
model_dt.fit(X_train, y_train)

# Evaluasi
prediksi_dt = model_dt.predict(X_test)
print("=== CLASSIFICATION REPORT: DECISION TREE (STANDARD K-MEANS) ===\n")
print(classification_report(y_test, prediksi_dt))

# 3. EKSTRAKSI ATURAN BISNIS
aturan_bisnis = export_text(model_dt, feature_names=fitur)
print("=== BUSINESS RULES (ATURAN LOGIKA PELANGGAN) ===\n")
print(aturan_bisnis)

# 4. EXPORT MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)
joblib.dump(model_dt, '../models/model_dt_classification.pkl')
print("\n[SUCCESS] Model Decision Tree diekspor ke '../models/model_dt_classification.pkl'")

=== CLASSIFICATION REPORT: DECISION TREE ===

              precision    recall  f1-score   support

           1       1.00      1.00      1.00         1
           2       0.97      0.96      0.96       295
           4       1.00      0.67      0.80         3
           5       0.98      0.98      0.98       568

    accuracy                           0.97       867
   macro avg       0.99      0.90      0.94       867
weighted avg       0.97      0.97      0.97       867

=== BUSINESS RULES (ATURAN LOGIKA PELANGGAN) ===

|--- Var8 <= 171.83
|   |--- Var1 <= 145.50
|   |   |--- Var4 <= 51009.55
|   |   |   |--- Var6 <= 766.50
|   |   |   |   |--- class: 5
|   |   |   |--- Var6 >  766.50
|   |   |   |   |--- class: 1
|   |   |--- Var4 >  51009.55
|   |   |   |--- Var7 <= 5.20
|   |   |   |   |--- class: 1
|   |   |   |--- Var7 >  5.20
|   |   |   |   |--- class: 4
|   |--- Var1 >  145.50
|   |   |--- Var6 <= 74.00
|   |   |   |--- Var3 <= 804.50
|   |   |   |   |--- class: 2
|   |   